# TDP-43 Single Molecule Tracking: Compartment Analysis
## Project: K136R Mutant vs. Wild-Type Diffusion Kinetics

### 1. Experimental Overview
This notebook processes TrackMate output files from 3-channel merged stacks. It separates
single-molecule trajectories based on their spatial location (Nucleus vs. Cytoplasm) using
binary mask intensities, then exports SPOT-On-compatible Regular CSV files.

**Acquisition Parameters:**
* **Pixel Size:** 0.1285 µm (positions exported in µm by TrackMate — no conversion needed)
* **Frame Interval:** 34.49 ms (~29 Hz) — time sourced directly from TrackMate `POSITION_T` column
* **Total Frames:** 5,000

### 2. Processing Logic
Channel 1 (TDP-43) spots are assigned to compartments using saturated binary mask intensities:

| Compartment | Criterion | Channel |
|---|---|---|
| Nuclear | `MEDIAN_INTENSITY_CH2 > 200` | CH2 = nuclear mask |
| Cytoplasmic | `MEDIAN_INTENSITY_CH3 > 200` | CH3 = cytoplasmic mask |

* Unlinked spots (no `TRACK_ID`) are removed — SPOT-On requires complete trajectories.
* Output format: **SPOT-On Regular CSV** (`trajectory`, `frame`, `x`, `y`, `t`)
* `trajectory` is cast to integer — required by the SPOT-On uploader.

### 3. TrackMate CSV Structure (for reference)

Row 0: machine column names  (TRACK_ID, POSITION_X, ...)
Row 1: human-readable names
Row 2: abbreviations
Row 3: units                  (microns, sec, counts, ...)
Row 4+: data


### 4. File Directory Configuration
*Paste your directory paths into the interactive boxes below and click **Process Batch**.*

In [ ]:
import pandas as pd
import os
import glob
import sys
import ipywidgets as widgets
from IPython.display import display, clear_output

# ---------------------------------------------------------------------------
# Detect environment
# ---------------------------------------------------------------------------

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# ---------------------------------------------------------------------------
# Processing logic (shared)
# ---------------------------------------------------------------------------
RENAME_MAP = {
    'TRACK_ID':   'trajectory',
    'FRAME':      'frame',
    'POSITION_X': 'x',
    'POSITION_Y': 'y',
    'POSITION_T': 't',
}
FINAL_COLS   = ['trajectory', 'frame', 'x', 'y', 't']
NUMERIC_COLS = ['TRACK_ID', 'FRAME', 'POSITION_X', 'POSITION_Y', 'POSITION_T',
                'MEDIAN_INTENSITY_CH2']

def write_spoton_csv(df, output_path):
    out_df = df.rename(columns=RENAME_MAP).copy()
    out_df['trajectory'] = out_df['trajectory'].astype(int)
    out_df['frame']      = out_df['frame'].astype(int)
    out_df[FINAL_COLS].to_csv(output_path, index=False)

def process_dataframe(df, base, output_dir, summary_rows):
    for col in NUMERIC_COLS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    df = df.dropna(subset=['TRACK_ID', 'FRAME', 'POSITION_X', 'POSITION_Y', 'POSITION_T'])

    cyto_df = df[df['MEDIAN_INTENSITY_CH2'].round() == 1].copy()
    nuc_df  = df[df['MEDIAN_INTENSITY_CH2'].round() == 2].copy()

    nuc_path  = os.path.join(output_dir, f"{base}_NUC.csv")
    cyto_path = os.path.join(output_dir, f"{base}_CYTO.csv")

    if not nuc_df.empty:
        write_spoton_csv(nuc_df, nuc_path)
    if not cyto_df.empty:
        write_spoton_csv(cyto_df, cyto_path)

    summary_rows.append({
        'File':         base,
        'Total_Tracks': df['TRACK_ID'].nunique(),
        'Nuc_Tracks':   nuc_df['TRACK_ID'].nunique(),
        'Cyto_Tracks':  cyto_df['TRACK_ID'].nunique(),
        'Nuc_Spots':    len(nuc_df),
        'Cyto_Spots':   len(cyto_df),
    })

    return nuc_path, cyto_path, nuc_df, cyto_df

# ---------------------------------------------------------------------------
# COLAB UI: file uploader
# ---------------------------------------------------------------------------
if IN_COLAB:
    upload_button = widgets.Button(description="Upload _allspots.csv files",
                                   button_style='success', icon='upload')
    out = widgets.Output()
    display(upload_button, out)

    def on_upload_clicked(b):
        with out:
            clear_output()
            print("📂 Select your *_allspots.csv files in the dialog...")
            uploaded = colab_files.upload()  # opens file picker

            if not uploaded:
                print("❌ No files uploaded.")
                return

            summary_rows = []
            output_dir = "/content/spoton_output"
            os.makedirs(output_dir, exist_ok=True)

            for filename, content in uploaded.items():
                if not filename.endswith('_allspots.csv'):
                    print(f"⚠️ Skipping {filename} — expected *_allspots.csv")
                    continue

                print(f"  → {filename}")
                base = filename.replace('_allspots.csv', '')

                import io
                raw = pd.read_csv(io.BytesIO(content), nrows=0)
                cols = raw.columns.tolist()
                df = pd.read_csv(io.BytesIO(content), skiprows=4, names=cols, low_memory=False)

                nuc_path, cyto_path, nuc_df, cyto_df = process_dataframe(
                    df, base, output_dir, summary_rows)

            print("\n✅ Done! Downloading output files...\n")
            display(pd.DataFrame(summary_rows))

            # Auto-download results
            for fname in os.listdir(output_dir):
                colab_files.download(os.path.join(output_dir, fname))

    upload_button.on_click(on_upload_clicked)

# ---------------------------------------------------------------------------
# LOCAL UI: folder path boxes (unchanged)
# ---------------------------------------------------------------------------
else:
    input_text  = widgets.Text(
        description="Source Folder:",
        placeholder="G:/Data/TrackMate_Outputs",
        layout={'width': '500px'}
    )
    output_text = widgets.Text(
        description="Save Folder:",
        placeholder="G:/Data/SpotOn_Ready",
        layout={'width': '500px'}
    )
    run_button = widgets.Button(description="Process Batch", button_style='success', icon='check')
    out        = widgets.Output()
    display(input_text, output_text, run_button, out)

    def on_button_clicked(b):
        with out:
            clear_output()
            input_dir  = input_text.value.strip().replace('"', '').replace('\\', '/')
            output_dir = output_text.value.strip().replace('"', '').replace('\\', '/')

            if not os.path.exists(input_dir):
                print(f"❌ Source folder not found: {input_dir}")
                return

            files = glob.glob(os.path.join(input_dir, "*_allspots.csv"))
            if not files:
                print("❓ No files matching *_allspots.csv found in that folder.")
                return

            os.makedirs(output_dir, exist_ok=True)
            print(f"🚀 Processing {len(files)} file(s)...\n")
            summary_rows = []

            for f in files:
                name = os.path.basename(f)
                base = name.replace('_allspots.csv', '')
                print(f"  → {name}")
                cols = pd.read_csv(f, nrows=0).columns.tolist()
                df   = pd.read_csv(f, skiprows=4, names=cols, low_memory=False)
                process_dataframe(df, base, output_dir, summary_rows)

            print("\n✅ Done. Upload output files to SPOT-On as 'Regular CSV'.\n")
            display(pd.DataFrame(summary_rows))

    run_button.on_click(on_button_clicked)

Text(value='', description='Source Folder:', layout=Layout(width='500px'), placeholder='G:/Data/TrackMate_Outp…

Text(value='', description='Save Folder:', layout=Layout(width='500px'), placeholder='G:/Data/SpotOn_Ready')

Button(button_style='success', description='Process Batch', icon='check', style=ButtonStyle())

Output()